# Household MPC Experiment

Dedicated paired baseline-versus-income-shock workflow. The implementation lives in `src/experiments/mpc.py`; the historical command remains available through `run_mpc_experiment.py`.

USES THE BASELINE CONFIG, no overrides

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import pandas as pd
from src.experiments.mpc import run_mpc_experiment
from src.mpc_analysis import MPCFilterConfig, plot_mpc_panel

## Inputs

In [2]:
RUN_MPC = True
COUNTRY = "FRA"
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
T_MAX = 30
SHOCK_PERIOD = 20
HORIZON_PERIODS = 4
SHOCK_FRACTION = 0.1
N_JOBS = 4
OUTPUT_DIR = "data/output_data/mpc-experiment"

filters = MPCFilterConfig(
    min_income=1000,
    income_top_quantile=0.99,
    consumption_income_ratio_quantiles=(0.01, 0.99),
    gross_wealth_top_quantile=0.999,
    head_age_min=25,
    head_age_max=75,
)

## Run

In [3]:
outputs = None
if RUN_MPC:
    outputs = run_mpc_experiment(
        seeds=SEEDS,
        t_max=T_MAX,
        shock_period=SHOCK_PERIOD,
        horizon_periods=HORIZON_PERIODS,
        shock_fraction=SHOCK_FRACTION,
        country_iso3=COUNTRY,
        output_dir=OUTPUT_DIR,
        n_jobs=N_JOBS,
        force_rebuild_data=True,
        apply_mpc_filters=True,
        mpc_filter_config=filters,
        stay_in_activity_bracket_only=False,
        stable_effective_labor_state_only=False,
        # Impact MPC on every household. The cumulative MPC is identified where the
        # labour-income gap between the two runs is at most 5% of the shock.
        labor_income_tolerance=0.05,
        # False: support counts the transfer as a resource. True: the support test ignores it.
        shock_exempt_from_support=True,
    )
else:
    print("Set RUN_MPC = True to run the experiment.")

{'seed': 1, 'country': 'FRA', 't_max': 30, 'raw_data_path': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/raw_data', 'output_dir': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment', 'data_cache': '/Users/andone/Documents/python_projects/INET-consumption/run_model/data/output_data/mpc-experiment/data.pkl'}
Configuration summary
{'productivity_growth': 'SimpleTFPGrowth',
 'productivity_investment_planner': 'TargetIntensityTFPInvestmentPlanner',
 'labour_market': {'name': 'ReservationWageBindingDefaultLabourMarketClearer',
                   'parameters': {'allow_switching_industries': True,
                                  'compare_with_normalised_inputs': True,
                                  'consider_reservation_wages': True,
                                  'firing_cost_fraction': 0.0,
                                  'firing_speed': 1.0,
                                  'hiring_cost_fraction': 0.0,
 

ValueError: Configuration mismatch for FRA. capital accounting settings must match between data_config and country_config. Mismatches: firms.capital_compensation_accounting_mode: data_config=surplus_pool, country_config=production_cost; firms.capital_depreciation_accounting_mode: data_config=eurostat_cfc, country_config=none

## Results and plots

In [ ]:
if outputs is not None:
    panel = pd.read_csv(outputs["analysis_dir"] / "household_mpc_panel.csv")
    summary = pd.read_csv(outputs["analysis_dir"] / "household_mpc_summary.csv")
    identified_cumulative = panel["cumulative_mpc_identified"].fillna(False).astype(bool)
    plot_panel = panel.copy()
    plot_panel["real_cmpc_4q"] = plot_panel["real_cmpc_4q"].where(identified_cumulative)
    plot_mpc_panel(
        plot_panel,
        variables=[
            "income",
            "net_wealth",
            "housing_tenure",
            "net_liquid_financial_assets",
            "illiquid_financial_assets",
            "housing_wealth",
        ],
        mpc_columns=["real_mpc_impact", "real_cmpc_4q"],
        plot_kind="box_only",
        y_range=(-0.5, 1.1),
        winsor_quantiles=(0, 1),
        # winsor_quantiles=(0.01, 0.99),
        winsor_mode="trim",
        output_dir=outputs["analysis_dir"] / "plots",
        output_format="png",
    )
    summary

### Slide plot: impact and 4-quarter MPC by income and liquidity quintile

In [ ]:
if outputs is not None:
    import matplotlib.pyplot as plt
    import numpy as np

    BLUE, ORANGE, INK, GREY = "#2B5C8A", "#B55A12", "#14213D", "#6A7179"
    quintiles = ["Q1", "Q2", "Q3", "Q4", "Q5"]
    bins = [
        ("income_bin", "Income quintile"),
        ("net_liquid_financial_assets_bin", "Net liquid assets quintile"),
    ]
    identified_cumulative = panel["cumulative_mpc_identified"].fillna(False).astype(bool)
    x = np.arange(len(quintiles))

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    for ax, (bin_col, label) in zip(axes, bins):
        impact = [panel.loc[panel[bin_col] == q, "real_mpc_impact"].dropna() for q in quintiles]
        cumulative = [
            panel.loc[(panel[bin_col] == q) & identified_cumulative, "real_cmpc_4q"].dropna() for q in quintiles
        ]
        box_impact = ax.boxplot(impact, positions=x - 0.18, widths=0.32, showfliers=False, patch_artist=True)
        box_cum = ax.boxplot(cumulative, positions=x + 0.18, widths=0.32, showfliers=False, patch_artist=True)
        for box, color in ((box_impact, BLUE), (box_cum, ORANGE)):
            for patch in box["boxes"]:
                patch.set(facecolor=color, alpha=0.75, edgecolor=color)
            for median in box["medians"]:
                median.set(color=INK, linewidth=2)
            for line in box["whiskers"] + box["caps"]:
                line.set(color=color)
        ax.axhline(0, color=GREY, linewidth=1)
        ax.set_xticks(x, quintiles)
        ax.set_xlabel(label)
        ax.set_title(label)
    axes[0].set_ylabel("MPC (real consumption per unit transfer)")
    axes[0].legend(
        [box_impact["boxes"][0], box_cum["boxes"][0]],
        ["Real impact MPC", "Real cumulative 4-quarter MPC"],
        frameon=False,
        loc="lower left",
    )
    fig.tight_layout()
    plots_dir = outputs["analysis_dir"] / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(plots_dir / "mpc_box_income_liquidity.png", dpi=130)

    impact_summary = panel.groupby("income_bin")["real_mpc_impact"].agg(impact_n="count", real_impact_mpc="mean")
    cumulative_summary = (
        panel.loc[identified_cumulative]
        .groupby("income_bin")["real_cmpc_4q"]
        .agg(cumulative_n="count", real_cumulative_mpc="mean")
    )
    print("Real MPC by income quintile (impact: filtered sample; cumulative: labour-income-identified sample):")
    print(impact_summary.join(cumulative_summary).round(2))
    print(
        "overall means:",
        {
            "real_mpc_impact": round(float(panel["real_mpc_impact"].mean()), 3),
            "real_cmpc_4q_identified": round(float(panel.loc[identified_cumulative, "real_cmpc_4q"].mean()), 3),
        },
    )
    print(
        "sample sizes:",
        {
            "impact": int(panel["real_mpc_impact"].count()),
            "cumulative_identified": int(panel.loc[identified_cumulative, "real_cmpc_4q"].count()),
            "filtered_panel": len(panel),
        },
    )